In [1]:
# from numba import cuda
import paicos as pa
import numpy as np
import cupy as cp
import turbocluster as tc
import math
from numba import cuda
import nvtx
import finufft

import cmasher as cmr
%matplotlib widget
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm, Normalize

pa.settings.strict_units = False



In [2]:
from paicos.util import get_index_of_radial_range as RadialRange

In [3]:
# A snapshot object
# snap = pa.Snapshot(pa.data_dir, 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/adiabatic-mhd/zoom4_ics_v1/output', 247)
# snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/scratch/lperrone/zoom-simulations-new-ics/halo_0003/tng/zoom12_ics_v1/output', 247)
# snap = pa.Snapshot('/llust21/cosmo-plasm/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   # 305, basename='snapshot')
snap = pa.Snapshot('/lustre/astro/berlok/zoom-simulations-arepo2/halo_0003/tng/zoom12/output', 
                   306, basename='snapshot')
center = snap.Cat.Group['GroupPos'][0]

radius = 1e2*snap.length

# widths = np.array([2e2, 2e2, 2e2], dtype=float)
widths = np.ones(3)*2*radius.value

Gauss = snap.uq('G')
arepo_length = snap.length
density_unit = snap['0_Density'].uq
L_box = (arepo_length*(widths[0]))

In [4]:
idx_sphere = RadialRange(snap["0_Coordinates"],
                       center, 0*radius, radius)

In [5]:
volumes = snap['0_Volume']

Attempting to get derived variable: 0_Volume...	[DONE]



In [6]:
snap['0_Masses'][idx_sphere] = volumes[idx_sphere]*density_unit
snap['0_Masses'][~idx_sphere] = 0

In [7]:
# smoothing_length = 0.05*cp.asnumpy(pe.tile.tile_widths).min()*arepo_length
smoothing_length = np.cbrt((snap["0_Volume"]) / (4.0 * np.pi / 3.0)).to('arepo_length')


In [8]:

pe = tc.PotentialEnergy(snap, center, widths, 
                        snap['0_Coordinates'], snap['0_Masses'], smoothing_length, npix=64)

In [9]:
pe.tile.mass_per_tile.shape

(64, 64, 64)

In [10]:
(4./3.)*np.pi*radius**3*density_unit

<PaicosQuantity 4188790.20478639 arepo_length3 arepo_density / small_h>

In [11]:
Mtot = np.sum(snap['0_Masses'])
print(Mtot)

4188519.3206029404 arepo_mass / small_h


In [12]:
np.sum(pe.index)

562196

In [13]:
np.sum(pe.tile.particles_per_tile)

array(562196)

In [14]:
np.histogram(pe.tile.particles_per_tile.flatten())

(array([86806, 86503, 53466, 23905,  8397,  2372,   559,   118,    15,
            3]),
 array([ 0. ,  1.1,  2.2,  3.3,  4.4,  5.5,  6.6,  7.7,  8.8,  9.9, 11. ]))

In [15]:
cp.asnumpy(pe.tile.tile_widths).min()*arepo_length

<PaicosQuantity 3.125 arepo_length small_a / small_h>

In [16]:
np.max(smoothing_length[pe.index])

<PaicosQuantity 2.29796074 arepo_length small_a / small_h>

In [17]:
angles = np.array([40, 25, 15, 10, 5, 2, 0])

num_cells_away = 0.5*np.sqrt(3)/np.tan(0.5*angles*np.pi/180)

/tmp/ipykernel_3976746/1320231423.py:3: RuntimeWarning: divide by zero encountered in divide
  num_cells_away = 0.5*np.sqrt(3)/np.tan(0.5*angles*np.pi/180)


In [18]:
num_cells_away

array([ 2.37938524,  3.90638815,  6.57811602,  9.89871566, 19.83524281,
       49.61456215,         inf])

In [22]:
pe.compute_potential(angle=0)

<PaicosQuantity -7.02791074 m3 arepo_mass2 / (arepo_length kg small_a small_h s2)>

In [ ]:
# for angle in angles:
#     U = pe.compute_potential(smoothing_length, angle=angle)
#     print(f'theta: {angle:.1f} \t', 'U: ', U)

In [ ]:
U_bench = []
runtime_bench = [] # in s
for angle in angles:
    U_tmp = []
    result = %timeit -n1 -r5 -o U_tmp.append(pe.compute_potential(angle=angle))
    U_bench.append(U_tmp[0])
    runtime_bench.append(result.average)

In [ ]:
runtime_bench

In [29]:
from numba import njit, prange
import cupy as cp
from numba import cuda
import math


@cuda.jit()
def cuda_potential(masses, pos, hsml, potential):

    Np =  masses.shape[0]
    i = cuda.grid(1)

    if (i < Np):
        for j in range(Np):
            h = max(hsml[i], hsml[j])
            m1m2 = masses[i]*masses[j]
            if (i==j):
                potential[i] -= m1m2*(14./5.)/h
            else:
                r = math.sqrt((pos[i][0] - pos[j][0])**2 +
                             (pos[i][1] - pos[j][1])**2 +
                             (pos[i][2] - pos[j][2])**2)
                potential_tmp = m1m2/r
                
                # if (r <= h):
                #     potential_tmp *= (0.5*r/h)*(3-(r/h)**2)
                u = r/h
                if (u < 0.5):
                    potential_tmp *= (14./5.)*u - (16./3.)*u**3 + (48./5.)*u**5 - (32./5.)*u**6
                elif (u >= 0.5 and u < 1):
                    potential_tmp *= 16.*u**4 - (48./5.)*u**5 + (32./15.)*u**6
                
                potential[i] -= potential_tmp
                


def direct_calculation_cuda(snap, positions, masses, smoothing_length):
    # snap_select = snap.select(index, parttype=0)

    Np = masses.shape[0]

    if isinstance(smoothing_length.value, np.ndarray):
        assert smoothing_length.shape == masses.shape
    else:
        smoothing_length = np.ones(Np) * smoothing_length

    mass = cp.array(masses.value)
    pos = cp.array(positions.value)
    hsml = cp.array(smoothing_length.value)
    potential = cp.zeros_like(mass)

    threadsperblock = 256
    blocks_1d = (Np + (threadsperblock - 1)) // threadsperblock
    

    cuda_potential[blocks_1d, threadsperblock](mass, pos, hsml, potential)

    potential = cp.asnumpy(cp.sum(potential))
    
    G = pa.astropy.constants.G
    potential = 0.5 * G * potential * snap.mass**2 / snap.length
    return potential
    

In [30]:
U_direct = []
# runtime_direct = [] # in s
result = %timeit -n1 -r1 -o U_direct.append(direct_calculation_cuda(snap, snap['0_Coordinates'][pe.index], snap['0_Masses'][pe.index], smoothing_length[pe.index]))

U_direct = U_direct[0]
runtime_direct = result.average

5.77 s ± 0 ns per loop (mean ± std. dev. of 1 run, 1 loop each)


In [31]:
U_direct

<PaicosQuantity -7.02791074 m3 arepo_mass2 / (arepo_length kg small_a small_h s2)>

In [ ]:
U_bench

In [20]:
G = pa.astropy.constants.G
# https://en.wikipedia.org/wiki/Gravitational_binding_energy
U_theory = -(3./5.)*G*Mtot**2/radius

In [32]:
U_theory.to('erg')

<PaicosQuantity -9.00734883e+68 erg / (small_a small_h)>

In [ ]:
relative_err = np.array([np.abs((U_bench[i]-U_direct)/U_direct).value for i in range(len(U_bench))])

In [ ]:


fig, ax = plt.subplots(2, 1, sharex=True)

ax[0].plot(angles, relative_err*100, ls='', marker='o', label='tiles')
ax[1].plot(angles, runtime_bench, ls='', marker='o', label='tiles')
ax[1].hlines(runtime_direct, xmin=-5, xmax=45, color='orange', ls='-', label='direct N^2')

ax[0].text(20, 0.02, f'direct N^2: {U_direct.value:.6f}')
ax[0].text(20, 0.015, f'theory: {U_theory.value:.6f}')

ax[0].set_xlim(xmin=-5, xmax=45)

ax[1].set_xlabel('angle')
ax[0].set_ylabel('relative err [%]')
ax[1].set_ylabel('timings [s]')
ax[0].legend()
ax[1].legend()

fig.suptitle('Benchmark of gravitational potential for uniform sphere')
plt.savefig('./uniform_sphere.pdf')

plt.show()